In [ ]:
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve
)

import matplotlib.pyplot as plt
import umap

# ---------- Paths ----------
DF_META_PATH = "df_meta.csv"
EMB_DIR = os.path.join("pretrained_feats", "msi_multitask_dinov2")
FEATS_PATH = os.path.join(EMB_DIR, "image_feats.npy")
INDEX_PATH = os.path.join(EMB_DIR, "index.csv")

# ---------- Load ----------
df_meta = pd.read_csv(DF_META_PATH, sep=None, engine="python")
emb = np.load(FEATS_PATH)
index_df = pd.read_csv(INDEX_PATH)

print("emb shape:", emb.shape)
print("index_df shape:", index_df.shape)
print("df_meta shape:", df_meta.shape)

# Merge via sample_path to be safe
df = df_meta.merge(index_df, on="sample_path", how="inner")
assert len(df) == emb.shape[0], "Mismatch between df rows and embedding rows after merge"


In [ ]:
# ---------- Define application filters ----------
TARGET_ORGANISM = "Homo sapiens"
TARGET_ORG_PART = "Kidney"
TARGET_POLARITY = "Negative"      # or "Positive" or None
TARGET_ANALYZER = None      # or None

df_app = df.copy()

# 1) Filter by organism / tissue
df_app = df_app[df_app["organism"] == TARGET_ORGANISM]
df_app = df_app[df_app["Organism_Part"] == TARGET_ORG_PART]

# 2) Optional: filter by polarity / analyzer
if TARGET_POLARITY is not None:
    df_app = df_app[df_app["polarity"] == TARGET_POLARITY]

if TARGET_ANALYZER is not None:
    df_app = df_app[df_app["analyzerType"] == TARGET_ANALYZER]

print("After basic filters:", df_app.shape)

# 3) Map Condition → Healthy / Diseased
def map_condition_to_binary(cond: str):
    if pd.isna(cond):
        return None
    cond = str(cond).lower()
    # you can tweak this mapping as you like
    healthy_terms = ["healthy", "wildtype", "wild-type", "control", "normal"]
    diseased_terms = ["tumor", "cancer", "diseased", "lesion"]

    if any(t in cond for t in healthy_terms):
        return "Healthy"
    if any(t in cond for t in diseased_terms):
        return "Diseased"
    return None  # drop NA / ambiguous

df_app["cond_binary"] = df_app["Condition"].apply(map_condition_to_binary)
df_app = df_app[df_app["cond_binary"].notna()]

print("After condition mapping:", df_app["cond_binary"].value_counts())


In [ ]:
# Align embeddings with filtered df_app via row indices
# After merge, df had same row order as emb; keep the subset by position.
keep_idx = df_app.index.to_numpy()
X = emb[keep_idx]
y = df_app["cond_binary"].to_numpy()
groups = df_app["dataset_id"].to_numpy()  # assuming you have dataset_id in df_meta

print("Final X shape:", X.shape)
print("Classes:", np.unique(y, return_counts=True))


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
)

out_dir = r"Y:\coskun-lab\Efe\MSI Foundation Model\hvD"
os.makedirs(out_dir, exist_ok=True)

# ---------------------------------------------------
# Group-wise split (same as you had)
# ---------------------------------------------------
gss = GroupShuffleSplit(test_size=0.2, n_splits=1, random_state=6740)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]
groups_train, groups_test = groups[train_idx], groups[test_idx]

print("Train size:", X_train.shape[0], "Test size:", X_test.shape[0])
print("Train datasets:", len(np.unique(groups_train)), "Test datasets:", len(np.unique(groups_test)))

# ---------------------------------------------------
# Class weights for imbalance
# ---------------------------------------------------
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train
)
cw = dict(zip(np.unique(y_train), class_weights))
print("Class weights:", cw)

# ---------------------------------------------------
# Encode labels as 0/1 (internal)
# ---------------------------------------------------
label_to_int = {cls: i for i, cls in enumerate(sorted(np.unique(y_train)))}
int_to_label = {i: cls for cls, i in label_to_int.items()}

y_train_int = np.array([label_to_int[c] for c in y_train])
y_test_int  = np.array([label_to_int[c] for c in y_test])

target_names = [int_to_label[i] for i in sorted(int_to_label.keys())]

# ---------------------------------------------------
# Linear probe (Logistic Regression)
# ---------------------------------------------------
clf = make_pipeline(
    StandardScaler(),
    LogisticRegression(
        max_iter=5000,
        class_weight={label_to_int[k]: v for k, v in cw.items()},
        solver="lbfgs"
    )
)

clf.fit(X_train, y_train_int)
y_pred_int = clf.predict(X_test)
y_proba = clf.predict_proba(X_test)[:, 1]  # prob for class "1" (e.g. Diseased)

# ---------------------------------------------------
# Metrics (text)
# ---------------------------------------------------
print("\n=== Classification report (test) ===")
report_str = classification_report(
    y_test_int, y_pred_int,
    target_names=target_names
)
print(report_str)

# Parse report as dict for plotting
report_dict = classification_report(
    y_test_int, y_pred_int,
    target_names=target_names,
    output_dict=True
)

# Confusion matrix
cm = confusion_matrix(y_test_int, y_pred_int)
print("Confusion matrix:\n", cm)

# ===================================================
# FIGURE 1: Per-class precision / recall / F1
# ===================================================
sns.set(style="whitegrid", context="talk")

# Build DF from report_dict
rows = []
for cls_name in target_names:
    metrics_cls = report_dict[cls_name]
    rows.append({
        "class": cls_name,
        "precision": metrics_cls["precision"],
        "recall": metrics_cls["recall"],
        "f1-score": metrics_cls["f1-score"],
    })

# Also add macro avg as a separate "class" if you want
rows.append({
    "class": "macro avg",
    "precision": report_dict["macro avg"]["precision"],
    "recall": report_dict["macro avg"]["recall"],
    "f1-score": report_dict["macro avg"]["f1-score"],
})

import pandas as pd
df_metrics = pd.DataFrame(rows)

df_melt = df_metrics.melt(
    id_vars="class",
    value_vars=["precision", "recall", "f1-score"],
    var_name="metric",
    value_name="score",
)

plt.figure()
ax = sns.barplot(
    data=df_melt,
    x="class",
    y="score",
    hue="metric"
)
ax.set_ylim(0, 1.05)
ax.set_ylabel("Score")
ax.set_xlabel("")
ax.set_title("Healthy vs Diseased linear probe\nper-class metrics", fontsize=11)
plt.legend(title="", fontsize=9, loc='lower right')

# annotate bars
for p in ax.patches:
    height = p.get_height()
    if not np.isnan(height):
        ax.annotate(f"{height:.2f}",
                    (p.get_x() + p.get_width() / 2., height),
                    ha="center", va="bottom",
                    fontsize=7, rotation=0, xytext=(0, 2),
                    textcoords="offset points")

plt.tight_layout()
OUT_FIG1 = "healthy_vs_diseased_metrics_barplot.png"
plt.savefig(os.path.join(out_dir, OUT_FIG1), dpi=300, bbox_inches="tight")
plt.show()
print(f"[SAVED] {OUT_FIG1}")

# ===================================================
# FIGURE 2: Confusion matrix heatmap
# ===================================================
plt.figure()
ax_cm = sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    cbar=False,
    xticklabels=target_names,
    yticklabels=target_names
)
ax_cm.set_xlabel("Predicted label")
ax_cm.set_ylabel("True label")
ax_cm.set_title("Healthy vs Diseased\nconfusion matrix", fontsize=11)

plt.tight_layout()
OUT_FIG2 = "healthy_vs_diseased_confusion_matrix.png"
plt.savefig(os.path.join(out_dir, OUT_FIG2), dpi=300, bbox_inches="tight")
plt.show()
print(f"[SAVED] {OUT_FIG2}")

# ===================================================
# FIGURE 3: Predicted probability distributions
# ===================================================
# Map back to string labels for nicer plotting
label_str_test = np.array([int_to_label[i] for i in y_test_int])

df_prob = pd.DataFrame({
    "y_true": label_str_test,
    "p_diseased": y_proba,
})

plt.figure()
ax_prob = sns.violinplot(
    data=df_prob,
    x="y_true",
    y="p_diseased",
    inner="quartile",
    cut=0
)
ax_prob.set_ylabel("Predicted probability of 'Diseased'")
ax_prob.set_xlabel("True label")
ax_prob.set_title("Healthy vs Diseased\nprobability distributions", fontsize=11)
ax_prob.set_ylim(-0.02, 1.02)

plt.tight_layout()
OUT_FIG3 = "healthy_vs_diseased_prob_distributions.png"
plt.savefig(os.path.join(out_dir, OUT_FIG3), dpi=300, bbox_inches="tight")
plt.show()
print(f"[SAVED] {OUT_FIG3}")


In [ ]:
# ---- NEW: save train / test / all sample lists for attribution ----

df_app_train = df_app.iloc[train_idx].copy()
df_app_test  = df_app.iloc[test_idx].copy()

# Output paths
test_list_path  = "healthy_vs_diseased_test_samples.csv"
train_list_path = "healthy_vs_diseased_train_samples.csv"
all_list_path   = "healthy_vs_diseased_all_samples.csv"

# Columns required by the attribution script
cols = ["sample_path", "Condition"]

# --- save test ---
df_app_test[cols].to_csv(test_list_path, index=False)
print(f"Saved GroupShuffleSplit TEST sample list → {test_list_path}  ({len(df_app_test)})")

# --- save train ---
df_app_train[cols].to_csv(train_list_path, index=False)
print(f"Saved GroupShuffleSplit TRAIN sample list → {train_list_path}  ({len(df_app_train)})")

# --- save all ---
df_app[cols].to_csv(all_list_path, index=False)
print(f"Saved ALL usable samples → {all_list_path}  ({len(df_app)})")

In [ ]:
# UMAP on all samples in this cohort
reducer = umap.UMAP(
    n_neighbors=15,
    min_dist=0.1,
    metric="cosine",
    random_state=6740
)
Z = reducer.fit_transform(X)

df_plot = df_app.copy()
df_plot["UMAP1"] = Z[:, 0]
df_plot["UMAP2"] = Z[:, 1]

# (a) Colored by condition
plt.figure()
for label, color in zip(["Healthy", "Diseased"], ["tab:blue", "tab:red"]):
    mask = df_plot["cond_binary"] == label
    plt.scatter(
        df_plot.loc[mask, "UMAP1"],
        df_plot.loc[mask, "UMAP2"],
        s=10, alpha=1.0, label=label
    )
plt.xlabel("UMAP1")
plt.ylabel("UMAP2")
plt.legend(loc="best")
plt.title("UMAP of MetaboFM embeddings (condition)")
plt.tight_layout()
plt.show()


### m/z attribution

In [ ]:
import os
import random
from dataclasses import dataclass
from typing import Dict, List, Optional
from tqdm import tqdm

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import timm

# ============================================================
# CONFIG
# ============================================================
SEED = 6740

@dataclass
class TrainConfig:
    df_meta_path: str = "df_meta.csv"

    # If sample_path is relative, prepend this base directory
    base_dir: str = ""  # e.g. "metaspace_images_dump"

    # Tasks (must match df columns)
    tasks: List[str] = None  # set below
    ignore_index: int = -100

    # Data loading
    batch_size: int = 64
    num_workers: int = 0

    # Model / training phases
    img_size: int = 224           # tile_h/tile_w in df_meta (you can crop later if you want)
    backbone_name: str = "vit_base_patch14_dinov2.lvd142m" #deit_base_distilled_patch16_224, vit_base_patch14_dinov2.lvd142m, vit_base_patch16_224.mae

    # Phase 1: heads only
    phase1_epochs: int = 8
    phase1_head_lr: float = 1e-3

    # Phase 2: fine-tune backbone
    phase2_epochs: int = 60
    phase2_head_lr: float = 1e-3
    phase2_backbone_lr: float = 1e-5
    unfreeze_last_n_blocks: int = 4

    # Checkpoint paths
    phase1_ckpt_path: str = os.path.join("checkpoints", "multitask_dinov2_phase1.pt")
    final_ckpt_path: str = os.path.join("checkpoints", "multitask_dinov2_final.pt")

    weight_decay: float = 0.01
    use_amp: bool = True
    early_stop_patience: int = 10

    # Multi-task loss weights
    task_weights: Dict[str, float] = None  # set below

    # SupCon on organism embeddings
    supcon_weight: float = 0.0
    supcon_temperature: float = 0.2

# ============================================================
# UTILS / SEEDING
# ============================================================
def set_seed(seed: int = 6740):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[INFO] Using device: {device}")

# ============================================================
# DATA LOADERS FROM DF_META
# ============================================================
def build_loaders_from_dfmeta(cfg: TrainConfig):
    df = pd.read_csv(cfg.df_meta_path, sep=None, engine="python")

    required_cols = {"sample_path", "channels", "split"}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"df_meta.csv missing columns: {missing}")

    # ============================================================
    # GLOBAL FILTER: drop classes with < MIN_CLASS_COUNT over ALL data
    # ============================================================
    MIN_CLASS_COUNT = 100
    keep_classes_by_task = {}

    for t in cfg.tasks:
        if t not in df.columns:
            print(f"[WARN] Task '{t}' not in df_meta; skipping in MIN_CLASS_COUNT filtering.")
            continue

        # Count only non-NA labels
        vc = df[t].dropna().value_counts()
        keep = vc[vc >= MIN_CLASS_COUNT].index.tolist()
        keep_classes_by_task[t] = set(keep)

        print(f"[INFO] Task '{t}': keeping {len(keep)} classes with ≥ {MIN_CLASS_COUNT} samples.")

    # Build a global mask: keep rows where every task label is either
    #  - NA (will become ignore_index) OR
    #  - in the kept class set for that task
    mask = np.ones(len(df), dtype=bool)
    for t, keep_set in keep_classes_by_task.items():
        if t not in df.columns or len(keep_set) == 0:
            # If no kept classes for this task, we don't enforce filtering on it
            continue
        col = df[t]
        mask &= (col.isna() | col.isin(keep_set))

    before_rows = len(df)
    df = df[mask].reset_index(drop=True)
    after_rows = len(df)
    print(f"[INFO] Global MIN_CLASS_COUNT filtering: {before_rows - after_rows} rows removed, {after_rows} remain.")

    # Optional: warn if any task ended up with <2 classes
    for t in cfg.tasks:
        if t in df.columns:
            n_classes = df[t].dropna().nunique()
            if n_classes < 2:
                print(f"[WARN] Task '{t}' has only {n_classes} classes after filtering. "
                      f"Consider removing this task from cfg.tasks.")

    # ============================================================
    # Now split by 'split' AFTER filtering
    # ============================================================
    df_train = df[df["split"] == "train"].reset_index(drop=True)
    df_val   = df[df["split"] == "val"].reset_index(drop=True) \
               if "val" in df["split"].unique() else pd.DataFrame(columns=df.columns)
    df_test  = df[df["split"] == "test"].reset_index(drop=True) \
               if "test" in df["split"].unique() else pd.DataFrame(columns=df.columns)

    print(f"[INFO] df_meta rows after filtering: train={len(df_train)} val={len(df_val)} test={len(df_test)}")

    # Determine global target channels and spatial size
    global_in_chans = int(df["channels"].max())
    target_h = cfg.img_size
    target_w = cfg.img_size
    print(f"[INFO] Using target shape: C={global_in_chans}, H={target_h}, W={target_w}")

    # Build train dataset (constructs label maps)
    train_ds = MSITilesFromNpy(
        df=df_train,
        base_dir=cfg.base_dir,
        task_cols=cfg.tasks,
        task_label_maps=None,
        ignore_index=cfg.ignore_index,
        target_channels=global_in_chans,
        target_h=target_h,
        target_w=target_w,
    )
    label_maps = train_ds.task_label_maps

    # Helper to build val/test with same label maps & target shapes
    def make_ds(sub_df):
        if len(sub_df) == 0:
            return None
        return MSITilesFromNpy(
            df=sub_df,
            base_dir=cfg.base_dir,
            task_cols=cfg.tasks,
            task_label_maps=label_maps,
            ignore_index=cfg.ignore_index,
            target_channels=global_in_chans,
            target_h=target_h,
            target_w=target_w,
        )

    val_ds = make_ds(df_val)
    test_ds = make_ds(df_test)

    train_loader = DataLoader(
        train_ds,
        batch_size=cfg.batch_size,
        shuffle=True,
        num_workers=cfg.num_workers,
        pin_memory=True,
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=cfg.batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=True,
    ) if val_ds is not None else None
    test_loader = DataLoader(
        test_ds,
        batch_size=cfg.batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=True,
    ) if test_ds is not None else None

    return train_loader, val_loader, test_loader, label_maps, global_in_chans

# ============================================================
# DATASET
# ============================================================
class MSITilesFromNpy(Dataset):
    """
    Dataset that reads tiles from .npy or .npz files using df_meta.csv rows.

    Standardizes all tiles to (target_channels, target_h, target_w)
    so that DataLoader can stack them into batches.

    Expects df to contain columns:
        sample_path, channels, and label columns (tasks).
    """

    def __init__(
        self,
        df: pd.DataFrame,
        base_dir: str,
        task_cols: List[str],
        task_label_maps: Optional[Dict[str, Dict[str, int]]] = None,
        ignore_index: int = -100,
        target_channels: Optional[int] = None,
        target_h: Optional[int] = None,
        target_w: Optional[int] = None,
    ):
        self.df = df.reset_index(drop=True)
        self.base_dir = base_dir
        self.task_cols = task_cols
        self.ignore_index = ignore_index

        # Target shapes (global)
        self.target_channels = target_channels
        self.target_h = target_h
        self.target_w = target_w

        # Build label maps if not provided
        self.task_label_maps = task_label_maps or {}
        for t in self.task_cols:
            if t not in self.df.columns:
                print(f"[WARN] Task column '{t}' not in df_meta; will be ignored.")
                self.task_label_maps[t] = {}
                continue

            if t not in self.task_label_maps:
                col = self.df[t].astype("string")
                uniques_raw = col.unique()
                cleaned = []
                for u in uniques_raw:
                    if pd.isna(u):
                        continue
                    s = str(u)
                    if s in ["<NA>", "nan", "NaN", "None"]:
                        continue
                    cleaned.append(s)
                uniques = sorted(set(cleaned))
                label_map = {cls: i for i, cls in enumerate(uniques)}
                self.task_label_maps[t] = label_map
                print(f"[INFO] Task '{t}' has {len(label_map)} classes.")

        # Just for logging
        self.channels_unique = sorted(self.df["channels"].unique())
        print(f"[INFO] Unique channel counts in this split: {self.channels_unique}")
        if len(self.channels_unique) > 1:
            print("[WARN] Multiple channel counts present; they will be padded/trimmed to "
                  f"C={self.target_channels}.")

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx: int):
        row = self.df.iloc[idx]
        sample_path = row["sample_path"]

        if self.base_dir and not os.path.isabs(sample_path):
            path = os.path.join(self.base_dir, sample_path)
        else:
            path = sample_path

        # ---- load npy / npz robustly ----
        loaded = np.load(path)
        if isinstance(loaded, np.lib.npyio.NpzFile):
            if "arr_0" in loaded.files:
                arr = loaded["arr_0"]
            else:
                arr = loaded[loaded.files[0]]
        else:
            arr = loaded

        arr = np.asarray(arr)

        # ---- sanitize NaNs / infs ----
        arr = np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)

        # optional: simple per-tile scaling to [0, 1] to avoid huge magnitudes
        vmax = arr.max()
        if vmax > 0:
            arr = arr / vmax

        # Sometimes there is an extra singleton dim, squeeze it
        if arr.ndim > 3:
            arr = np.squeeze(arr)

        if arr.ndim != 3:
            raise ValueError(f"Expected (C,H,W) or (H,W,C) array, got shape {arr.shape} at {path}")

        expected_c = int(row["channels"])

        # Make sure channels are the first dim -> (C,H,W)
        if arr.shape[0] == expected_c:
            # already (C,H,W)
            pass
        elif arr.shape[-1] == expected_c:
            # probably (H,W,C) -> transpose
            arr = np.moveaxis(arr, -1, 0)
        else:
            # best-effort: assume first dim is channels
            pass

        C, H, W = arr.shape

        # ---- channel padding/truncation to target_channels ----
        if self.target_channels is not None:
            C_target = self.target_channels
            if C < C_target:
                pad = np.zeros((C_target - C, H, W), dtype=arr.dtype)
                arr = np.concatenate([arr, pad], axis=0)
            elif C > C_target:
                arr = arr[:C_target, :, :]
            C, H, W = arr.shape

        x = torch.from_numpy(arr).float()  # (C,H,W)

        # ---- spatial resize to (target_h, target_w) ----
        if self.target_h is not None and self.target_w is not None:
            x = x.unsqueeze(0)  # (1,C,H,W)
            x = F.interpolate(
                x,
                size=(self.target_h, self.target_w),
                mode="nearest"
            )
            x = x.squeeze(0)  # (C,H,W)

        labels = {}
        for t in self.task_cols:
            if t not in self.df.columns:
                labels[t] = self.ignore_index
                continue

            val = row[t]
            if pd.isna(val):
                labels[t] = self.ignore_index
            else:
                s = str(val)
                lm = self.task_label_maps.get(t, {})
                labels[t] = lm.get(s, self.ignore_index)

        labels["sample_path"] = row["sample_path"]

        return x, labels

# ============================================================
# MODEL: BACKBONE + MULTI-TASK HEADS
# ============================================================
class MultiTaskViT(nn.Module):
    def __init__(
        self,
        backbone_name: str,
        in_chans: int,
        task_label_maps: Dict[str, Dict[str, int]],
        img_size: int,
    ):
        super().__init__()
        self.backbone = timm.create_model(
            backbone_name,
            pretrained=True,
            num_classes=0,   # return embedding
            in_chans=in_chans,
            img_size=img_size,
        )

        # disable strict size check if present
        if hasattr(self.backbone, "patch_embed") and hasattr(self.backbone.patch_embed, "strict_img_size"):
            self.backbone.patch_embed.strict_img_size = False

        embed_dim = self.backbone.num_features

        self.tasks = list(task_label_maps.keys())
        self.heads = nn.ModuleDict()
        for t, lm in task_label_maps.items():
            num_classes = len(lm)
            if num_classes <= 1:
                num_classes = max(num_classes, 1)
            self.heads[t] = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        z = self.backbone(x)  # (B, D)
        logits = {t: head(z) for t, head in self.heads.items()}
        return z, logits
    
def build_meta_name_maps(df_meta: pd.DataFrame, base_dir: str = ""):
    """
    Returns:
      maps: dict with path_map/base_map/id_map/index_map
      sig_index: {(channels,tile_h,tile_w): [row_indices...]}
      per_ds_buckets: {dataset_id: [row_indices...]}
      dfm: df_meta with integer-safe cols
    """
    path_map, base_map, id_map, index_map = {}, {}, {}, {}

    # Normalize and prebuild quick maps
    if "sample_path" in df_meta.columns:
        for _, row in df_meta.iterrows():
            raw = str(row["sample_path"])
            norm = _norm_path_like(raw, base_dir)
            safe = _compose_row_name(row)
            path_map[norm] = safe
            base_map[_basename_stem(raw)] = safe

    if "filename" in df_meta.columns:
        for _, row in df_meta.iterrows():
            v = str(row["filename"])
            base_map[_basename_stem(v)] = _compose_row_name(row)

    if "dataset_id" in df_meta.columns:
        unique_ids = df_meta["dataset_id"].astype(str).unique().tolist()
        for v in unique_ids:
            id_map[v] = make_safe_name(v)

    # Always index_map for df index
    df_reset = df_meta.reset_index()
    for _, row in df_reset.iterrows():
        idx = int(row["index"])
        index_map[idx] = _compose_row_name(row)

    # Signature buckets to help last-resort matching
    sig_index = collections.defaultdict(list)
    per_ds_buckets = collections.defaultdict(list)

    # Use safe ints; fillna to avoid float NaN
    dfm = df_meta.copy()
    for col in ["channels", "tile_h", "tile_w", "tile_r", "tile_c"]:
        if col in dfm.columns:
            dfm[col] = pd.to_numeric(dfm[col], errors="coerce").fillna(-1).astype(int)

    for idx, row in dfm.iterrows():
        ch = int(row.get("channels", -1))
        th = int(row.get("tile_h", -1))
        tw = int(row.get("tile_w", -1))
        sig_index[(ch, th, tw)].append(idx)
        ds = str(row.get("dataset_id", ""))
        per_ds_buckets[ds].append(idx)

    return (
        {"path_map": path_map, "base_map": base_map, "id_map": id_map, "index_map": index_map},
        sig_index,
        per_ds_buckets,
        dfm,
    )

class DFMetaHeuristicAssigner:
    """
    Deterministic name assignment when loader provides no usable identifiers.
    """
    def __init__(self, df_meta, sig_index, per_ds_buckets):
        self.df_meta = df_meta
        self.sig_index = {k: list(v) for k, v in sig_index.items()}
        self.per_ds_buckets = {k: list(v) for k, v in per_ds_buckets.items()}
        self.global_q = list(range(len(df_meta)))
        self.used = set()

    def _pop_from_list(self, L):
        while L:
            idx = L.pop(0)
            if idx not in self.used:
                self.used.add(idx)
                return idx
        return None

    def consume_by_signature(self, channels, tile_h, tile_w):
        key = (int(channels), int(tile_h), int(tile_w))
        L = self.sig_index.get(key, [])
        if not L: return None
        idx = self._pop_from_list(L)
        self.sig_index[key] = L
        return idx

    def consume_by_dataset(self, dataset_id):
        L = self.per_ds_buckets.get(str(dataset_id), [])
        if not L: return None
        idx = self._pop_from_list(L)
        self.per_ds_buckets[str(dataset_id)] = L
        return idx

    def consume_global(self):
        return self._pop_from_list(self.global_q)
    
def _compose_row_name(row: pd.Series) -> str:
    # e.g., 2024-02-06_13h58m18s__r000_c000 (zero-padded)
    ds = str(row.get("dataset_id", "ds"))
    r  = int(row.get("tile_r", 0))
    c  = int(row.get("tile_c", 0))
    return f"{make_safe_name(ds)}__r{r:03d}_c{c:03d}"

def enable_vit_attention_capture(vit):
    for blk in vit.blocks:
        attn = blk.attn
        if hasattr(attn, "_orig_forward"):
            continue
        attn._orig_forward = attn.forward

        def forward_with_capture(self, x, attn_mask=None, **kwargs):
            B, N, C = x.shape
            qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, C // self.num_heads).permute(2, 0, 3, 1, 4)
            q, k, v = qkv[0], qkv[1], qkv[2]
            q = q * self.scale
            attn_mat = q @ k.transpose(-2, -1)

            if hasattr(self, "rel_pos") and callable(getattr(self, "rel_pos")):
                try:
                    attn_mat = attn_mat + self.rel_pos(q)
                except Exception:
                    pass
            elif hasattr(self, "attn_bias") and self.attn_bias is not None:
                try:
                    bias = self.attn_bias
                    if bias.dim() == 3:
                        bias = bias.unsqueeze(0)
                    attn_mat = attn_mat + bias.to(attn_mat.dtype).to(attn_mat.device)
                except Exception:
                    pass

            if attn_mask is not None:
                attn_mat = attn_mat + attn_mask

            attn_soft = attn_mat.softmax(dim=-1)
            self.last_attn = attn_soft.detach()
            return self._orig_forward(x, attn_mask=attn_mask, **kwargs)

        attn.forward = forward_with_capture.__get__(attn, attn.__class__)

def collect_block_attentions(vit):
    attns = []
    for blk in vit.blocks:
        if not hasattr(blk.attn, "last_attn"):
            raise RuntimeError("Run a forward pass after enable_vit_attention_capture(...)")
        attns.append(blk.attn.last_attn)
    return attns

def attention_rollout(attn_list, head_fusion="mean", discard_ratio=0.9):
    device = attn_list[0].device
    B, _, T, _ = attn_list[0].shape
    I = torch.eye(T, device=device).unsqueeze(0).expand(B, T, T)
    A = I.clone()
    for attn in attn_list:
        if head_fusion == "mean":
            a = attn.mean(dim=1)
        elif head_fusion == "max":
            a = attn.max(dim=1)[0]
        else:
            raise ValueError

        # Optional: discard lowest attention entries
        if discard_ratio > 0:
            flat = a.view(B, -1)
            _, idx = flat.topk(k=int(flat.size(1) * (1 - discard_ratio)), dim=1)
            mask = torch.zeros_like(flat)
            mask.scatter_(1, idx, 1)
            a = a * mask.view_as(a)

        a = a + I
        a = a / a.sum(dim=-1, keepdim=True)
        A = torch.bmm(a, A)

    cls_to_all = A[:, 0, :]
    cls_to_patches = cls_to_all[:, 1:]
    return cls_to_patches

def patches_to_heatmap(scores, vit, img_h, img_w):
    B, Np = scores.shape
    if hasattr(vit.patch_embed, "grid_size") and vit.patch_embed.grid_size is not None:
        gh, gw = vit.patch_embed.grid_size
    else:
        side = int(math.sqrt(Np))
        gh, gw = side, side
    maps = scores.view(B, 1, gh, gw)
    maps = maps - maps.amin(dim=(2,3), keepdim=True)
    maps = maps / (maps.amax(dim=(2,3), keepdim=True) + 1e-6)
    maps_up = F.interpolate(maps, size=(img_h, img_w), mode="bilinear", align_corners=False)
    return maps_up  # [B,1,H,W]

def channel_attribution_input_grad(model, x, task_name, class_targets=None, use_inputxgrad=True):
    model.zero_grad(set_to_none=True)
    x = x.requires_grad_(True)
    with torch.no_grad():
        _, logits = model(x)
    if task_name not in logits:
        raise KeyError(f"Task '{task_name}' not found in model heads. Available: {list(logits.keys())}")
    logit_task = logits[task_name]
    if class_targets is None:
        class_targets = logit_task.argmax(dim=1)
    _, logits = model(x)
    pick = logits[task_name][torch.arange(x.size(0), device=x.device), class_targets]
    target = pick.sum()
    target.backward()
    sal = torch.abs(x.grad * x) if use_inputxgrad else torch.abs(x.grad)
    return sal.detach()

def patch_embed_channel_norms(vit):
    W = vit.patch_embed.proj.weight  # [D, C, p, p]
    ch_norm = (W ** 2).sum(dim=(0,2,3)).sqrt()
    ch_norm = (ch_norm - ch_norm.min()) / (ch_norm.max() - ch_norm.min() + 1e-6)
    return ch_norm.detach()

def _minmax01(a):
    a = a.astype(np.float32, copy=False)
    mn, mx = np.min(a), np.max(a)
    if mx <= mn + 1e-12:
        return np.zeros_like(a)
    return (a - mn) / (mx - mn + 1e-12)

def save_pca_rgb(img_chw, out_png, title=None):
    C, H, W = img_chw.shape
    X = img_chw.reshape(C, H*W).T
    X = X - X.mean(axis=0, keepdims=True)
    U, S, Vt = np.linalg.svd(X, full_matrices=False)
    PCs = X @ Vt.T[:, :3]
    PCs_norm = np.zeros_like(PCs, dtype=np.float32)
    for i in range(3):
        PCs_norm[:, i] = _minmax01(PCs[:, i])
    rgb = PCs_norm.reshape(H, W, 3)
    plt.figure(figsize=(4,4))
    plt.imshow(rgb); plt.axis("off")
    if title: plt.title(title)
    plt.tight_layout(); plt.savefig(out_png, dpi=200); plt.close()

def save_saliency_heatmap(
    saliency_chw,
    out_png,
    title=None,
    cmap_name="jet",
    attn_hw=None,
    save_joint=True,
    joint_suffix="_salxattn"
):
    # 1) Aggregate channels with SUM or L2, not max
    sal = np.abs(saliency_chw).sum(axis=0)  # or np.linalg.norm(saliency_chw, axis=0)

    # 2) Clip low saliency to kill background noise
    lo, hi = np.percentile(sal, [70, 99.9])  # keep only top 30% range
    sal = np.clip(sal, lo, hi)

    # 3) Normalize after clipping
    sal = _minmax01(sal)

    # --- Saliency-only ---
    plt.figure(figsize=(4,4))
    im = plt.imshow(sal, cmap=cmap_name)
    plt.axis("off")
    if title:
        plt.title(title)
    cbar = plt.colorbar(im, fraction=0.046, pad=0.04)
    cbar.ax.tick_params(labelsize=8)
    plt.tight_layout()
    plt.savefig(out_png, dpi=200)
    plt.close()

    # --- Saliency × Attention (if requested) ---
    if (attn_hw is not None) and save_joint:
        if attn_hw.shape != saliency_chw.shape[1:]:
            from skimage.transform import resize
            attn_hw = resize(attn_hw, saliency_chw.shape[1:], order=1, mode="reflect", anti_aliasing=True)

        # Use the *processed* sal, not raw sum
        joint = sal * attn_hw
        joint = _minmax01(joint)

        out_png_joint = out_png.replace(".png", f"{joint_suffix}.png")

        plt.figure(figsize=(4,4))
        im = plt.imshow(joint, cmap=cmap_name)
        plt.axis("off")
        if title:
            plt.title(title + " × Attn")
        cbar = plt.colorbar(im, fraction=0.046, pad=0.04)
        cbar.ax.tick_params(labelsize=8)
        plt.tight_layout()
        plt.savefig(out_png_joint, dpi=200)
        plt.close()

def save_channel_viridis(img_hw, out_png, title=None):
    im = _minmax01(img_hw)
    plt.figure(figsize=(4,4))
    plt.imshow(im, cmap="viridis"); plt.axis("off")
    if title: plt.title(title)
    plt.tight_layout(); plt.savefig(out_png, dpi=200); plt.close()

def spatio_spectral_scores(saliency_bchw, attn_b1hw):
    B, C, H, W = saliency_bchw.shape
    attn = attn_b1hw
    attn_sum = attn.sum(dim=(2,3), keepdim=True).clamp_min(1e-6)
    attn_norm = attn / attn_sum
    weighted = saliency_bchw * attn_norm
    scores = weighted.sum(dim=(2,3))
    return scores

In [ ]:
"""
Healthy vs Diseased m/z attribution using spatio–spectral attention.

Assumes the following are already available (either in this file or imported):
- TrainConfig
- MultiTaskViT
- MSITilesFromNpy
- build_loaders_from_dfmeta
- enable_vit_attention_capture
- collect_block_attentions
- attention_rollout
- patches_to_heatmap
- channel_attribution_input_grad
- patch_embed_channel_norms
- spatio_spectral_scores
- save_pca_rgb
- save_saliency_heatmap
- save_channel_viridis
- build_meta_name_maps, DFMetaHeuristicAssigner
"""

import os, json, re, collections, math, glob
from typing import Optional

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from tqdm import tqdm
from torch.utils.data import DataLoader

# =======================
# USER CONFIG
# =======================
TASK_FOR_ATTRIB       = "Condition"
USE_INPUTxGRAD        = True
BATCHES_TO_PROCESS    = None      # None = process all batches; or set an int limit
SAVE_TOPK_CHANNELS    = 10
SAVE_TOPK_CHANNEL_IMS = True
DF_META_PATH          = "df_meta.csv"

# The list of *held-out* samples for Healthy vs Diseased
HVD_TEST_LIST         = "healthy_vs_diseased_all_samples.csv"

# Roots used to resolve .npz filenames to actual files
DATA_ROOTS = [
    r"Y:\coskun-lab\Efe\MSI Foundation Model",
    r"Y:\coskun-lab\Efe\MSI Foundation Model\metaspace_images_dump\msi_fm_samples",
    os.getcwd(),
]
LIKELY_SUBFOLDERS = [os.path.join("metaspace_images_dump", "msi_fm_samples3"), ""]

_MZ_KEYS = ["mz", "mzs", "mz_axis", "mass", "mass_axis", "MZ", "MZ_axis"]

# =======================
# Basic helpers (paths, names)
# =======================
def _safe_tag(s: str) -> str:
    return os.path.splitext(os.path.basename(s))[0].replace(".", "_").replace(":", "_")

def make_safe_name(s: str) -> str:
    s = os.path.splitext(os.path.basename(str(s)))[0]
    s = s.replace(" ", "_")
    for bad in ['\\', '/', ':', '*', '?', '"', '<', '>', '|']:
        s = s.replace(bad, '_')
    return s

def make_outdir(cfg):
    run_tag = f"{_safe_tag(cfg.backbone_name)}__{_safe_tag(cfg.final_ckpt_path)}"
    out_dir = os.path.join("fm_ssl_run", "attn_attrib_hvD", run_tag)
    os.makedirs(out_dir, exist_ok=True)
    return out_dir

def _norm_path_like(s: str, base_dir: str = "") -> str:
    p = s
    try:
        if base_dir and not os.path.isabs(p):
            p = os.path.join(base_dir, p)
        p = os.path.normpath(p)
        p = os.path.normcase(p)
    except Exception:
        pass
    return p

def _basename_stem(p: str) -> str:
    return os.path.splitext(os.path.basename(str(p)))[0]

def _compose_row_name(row: pd.Series) -> str:
    # e.g., 2024-02-06_13h58m18s__r000_c000 (zero-padded)
    ds = str(row.get("dataset_id", "ds"))
    r  = int(row.get("tile_r", 0))
    c  = int(row.get("tile_c", 0))
    return f"{make_safe_name(ds)}__r{r:03d}_c{c:03d}"

# =======================
# NPZ / m/z helpers
# =======================
def resolve_npz_path(sp: str) -> Optional[str]:
    """Resolve a .npz path robustly using DATA_ROOTS and common subfolders."""
    if sp is None:
        return None
    sp_norm = os.path.normpath(str(sp))
    base = os.path.basename(sp_norm)
    base_noext, ext = os.path.splitext(base)

    # absolute direct
    if os.path.isabs(sp_norm) and os.path.exists(sp_norm):
        return sp_norm

    # roots + original relative
    for root in DATA_ROOTS:
        cand = os.path.normpath(os.path.join(root, sp_norm))
        if os.path.exists(cand):
            return cand

    # roots + likely subfolders + basename
    for root in DATA_ROOTS:
        for sub in LIKELY_SUBFOLDERS:
            cand2 = os.path.normpath(os.path.join(root, sub, base))
            if os.path.exists(cand2):
                return cand2
            if ext == "" and os.path.exists(cand2 + ".npz"):
                return cand2 + ".npz"

    # recursive basename search
    for root in DATA_ROOTS:
        pat = os.path.join(os.path.normpath(root), "**", base)
        hits = glob.glob(pat, recursive=True)
        if not hits and ext == "":
            hits = glob.glob(pat + ".npz", recursive=True)
        if hits:
            return os.path.normpath(hits[0])

    return None

def load_mz_axis_only(npz_path: str) -> Optional[np.ndarray]:
    """Return m/z axis as float vector or None on failure."""
    try:
        with np.load(npz_path, allow_pickle=True) as npz:
            mz_key = next((k for k in _MZ_KEYS if k in npz.files), None)
            if mz_key is None:
                return None
            mz = np.array(npz[mz_key], dtype=float).ravel()
            return mz
    except Exception:
        return None

def _fmt_mz_for_filename(m: float) -> str:
    # e.g., 734.22 -> "734p22" to be filesystem-safe and sortable
    return f"{float(m):.2f}".replace(".", "p")

# =======================
# Condition → Healthy/Diseased mapping
# =======================
def map_condition_to_binary(cond: str) -> Optional[str]:
    if cond is None or (isinstance(cond, float) and np.isnan(cond)):
        return None
    c = str(cond).lower()
    healthy_terms  = ["healthy", "control", "wildtype", "wild-type", "normal"]
    diseased_terms = ["tumor", "cancer", "diseased", "lesion"]

    if any(t in c for t in healthy_terms):
        return "Healthy"
    if any(t in c for t in diseased_terms):
        return "Diseased"
    return None

# Regex to parse dataset_id__r###_c###
_rowname_re = re.compile(r"^(?P<ds>.+?)__r(?P<r>\d+)_c(?P<c>\d+)$")

# =======================
# MAIN
# =======================
def main():
    # ------------------ Config / loaders / model ------------------
    cfg = TrainConfig()
    if cfg.tasks is None:
        cfg.tasks = [
            "organism",
            "polarity",
            "Organism_Part",
            "Condition",
            "analyzerType",
            "ionisationSource",
        ]
    if cfg.task_weights is None:
        cfg.task_weights = {
            "organism":        1.0,
            "Organism_Part":   1.0,
            "polarity":        0.8,
            "analyzerType":    0.5,
            "ionisationSource":0.5,
            "Condition":       0.3,
        }

    out_dir = make_outdir(cfg)
    os.makedirs(out_dir, exist_ok=True)
    with open(os.path.join(out_dir, "attrib_config_healthy_vs_diseased.json"), "w") as f:
        json.dump(cfg.__dict__, f, indent=2)

    # ------------------ Healthy vs Diseased sample list ------------------
    if not os.path.isfile(HVD_TEST_LIST):
        raise FileNotFoundError(
            f"Sample list '{HVD_TEST_LIST}' not found. "
            f"Run the GroupShuffleSplit script first to create it."
        )

    hvd_df = pd.read_csv(HVD_TEST_LIST)
    if "sample_path" not in hvd_df.columns:
        raise ValueError(f"'{HVD_TEST_LIST}' must contain a 'sample_path' column.")

    hvd_paths      = set(str(p) for p in hvd_df["sample_path"].dropna().astype(str))
    hvd_safe_names = {make_safe_name(p) for p in hvd_paths}
    print(f"[INFO] Loaded {len(hvd_paths)} sample paths for Healthy-vs-Diseased attribution.")

    # ------------------ df_meta + name maps ------------------
    df_meta      = None
    df_meta_int  = None
    meta_maps    = {}
    sig_index    = {}
    per_ds_buckets = {}
    assigner     = None
    safe_name_to_sample_path = {}

    if os.path.isfile(DF_META_PATH):
        df_meta = pd.read_csv(DF_META_PATH)
        meta_maps, sig_index, per_ds_buckets, df_meta_int = build_meta_name_maps(
            df_meta, base_dir=getattr(cfg, "base_dir", "")
        )
        assigner = DFMetaHeuristicAssigner(df_meta_int, sig_index, per_ds_buckets)

        if "sample_path" in df_meta.columns:
            for sp in df_meta["sample_path"].dropna().astype(str):
                safe_name_to_sample_path[make_safe_name(sp)] = sp

    # ------------------ Build loaders (for label_maps + in_chans) ------------------
    # This still constructs full train/val/test splits with MIN_CLASS_COUNT filtering.
    train_loader, val_loader, test_loader, label_maps, in_chans = build_loaders_from_dfmeta(cfg)

    # ------------------ Build dedicated HVD dataset/loader from df_meta ------------------
    if df_meta is None:
        raise RuntimeError("DF_META_PATH not found or could not be read, cannot build HVD dataset.")

    # Match by exact path or safe basename
    sp_col = df_meta["sample_path"].astype(str)
    mask = sp_col.isin(hvd_paths) | sp_col.map(make_safe_name).isin(hvd_safe_names)
    df_hvd = df_meta[mask].reset_index(drop=True)

    print(f"[INFO] Found {len(df_hvd)} matching HVD rows in df_meta (all splits).")
    if len(df_hvd) < len(hvd_paths):
        missing = sorted(hvd_paths - set(df_hvd["sample_path"].astype(str)))
        if missing:
            print("[WARN] Some HVD sample paths in CSV could not be found in df_meta:")
            for m in missing:
                print("   MISSING in df_meta:", m)

    if df_hvd.empty:
        raise RuntimeError("No HVD rows found in df_meta; nothing to attribute.")

    hvd_ds = MSITilesFromNpy(
        df=df_hvd,
        base_dir=getattr(cfg, "base_dir", ""),
        task_cols=cfg.tasks,
        task_label_maps=label_maps,            # reuse label maps from training build
        ignore_index=cfg.ignore_index,
        target_channels=in_chans,
        target_h=cfg.img_size,
        target_w=cfg.img_size,
    )

    hvd_loader = DataLoader(
        hvd_ds,
        batch_size=cfg.batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=True,
    )

    # invert label maps so we can go from int → string label
    inv_label_maps = {
        task: {idx: lbl for lbl, idx in lm.items()}
        for task, lm in label_maps.items()
    }

    # ------------------ Model ------------------
    model = MultiTaskViT(
        backbone_name=cfg.backbone_name,
        in_chans=in_chans,
        task_label_maps=label_maps,
        img_size=cfg.img_size,
    ).to(torch.device("cuda" if torch.cuda.is_available() else "cpu"))

    ckpt_path = cfg.final_ckpt_path if os.path.isfile(cfg.final_ckpt_path) else cfg.phase1_ckpt_path
    ckpt = torch.load(ckpt_path, map_location="cpu")
    state_dict = ckpt.get("model_state", ckpt)
    model.load_state_dict(state_dict)
    model.eval()
    device = next(model.parameters()).device

    enable_vit_attention_capture(model.backbone)
    loader = hvd_loader  # <- attribution only on HVD samples

    # ------------------ Static channel weights from patch embedding ------------------
    ch_weight_norm = patch_embed_channel_norms(model.backbone).cpu().numpy()
    np.save(os.path.join(out_dir, "channel_weight_norms.npy"), ch_weight_norm)
    pd.DataFrame(
        {"channel": np.arange(len(ch_weight_norm)), "weight_norm": ch_weight_norm}
    ).to_csv(os.path.join(out_dir, "channel_weight_norms.csv"), index=False)

    samples_root = os.path.join(out_dir, "samples")
    os.makedirs(samples_root, exist_ok=True)

    # helper: map sample folder name to sample_path via df_meta
    def name_to_sample_path(sample_folder_name: str) -> Optional[str]:
        sp = safe_name_to_sample_path.get(sample_folder_name)
        if sp:
            return sp
        if df_meta_int is not None:
            m = _rowname_re.match(sample_folder_name)
            if m:
                ds = m.group("ds")
                r  = int(m.group("r"))
                c  = int(m.group("c"))
                hits = df_meta_int.index[
                    (df_meta_int["dataset_id"].astype(str).map(make_safe_name) == ds) &
                    (df_meta_int["tile_r"] == r) &
                    (df_meta_int["tile_c"] == c)
                ].tolist()
                if hits and "sample_path" in df_meta.columns:
                    return str(df_meta.loc[hits[0], "sample_path"])
        return None

    # strong name resolver
    def resolve_names(labels_dict, B, batch_idx, x_shape):
        names = []
        for b in range(B):
            sp = labels_dict["sample_path"]
            sp = sp[b] if isinstance(sp, (list, tuple)) else sp
            sp = str(sp)

            # extract last part of the sample_path (file stem)
            stem = os.path.splitext(os.path.basename(sp))[0]
            names.append(make_safe_name(stem))
        return names

    # ------------------ ATTRIBUTION LOOP ------------------
    all_rows = []
    processed_batches = 0

    for batch_idx, (x, labels_dict) in enumerate(loader):
        if BATCHES_TO_PROCESS is not None and processed_batches >= BATCHES_TO_PROCESS:
            break

        x = x.to(device, non_blocking=True)
        B, C, H, W = x.shape

        # forward to get attentions
        with torch.no_grad():
            _ = model(x)
        attn_list = collect_block_attentions(model.backbone)
        cls_roll  = attention_rollout(attn_list, head_fusion="mean", discard_ratio=0.0)
        attn_maps = patches_to_heatmap(cls_roll, model.backbone, H, W)

        # saliency for Condition task
        saliency = channel_attribution_input_grad(
            model, x.clone(), task_name=TASK_FOR_ATTRIB,
            class_targets=None, use_inputxgrad=USE_INPUTxGRAD
        )

        # spatio–spectral scores per channel
        ss_scores = spatio_spectral_scores(saliency, attn_maps).cpu().numpy()

        # optional batch-wise artifacts
        np.save(
            os.path.join(out_dir, f"attn_maps_batch{batch_idx:04d}.npy"),
            attn_maps.squeeze(1).cpu().numpy(),
        )
        np.save(
            os.path.join(out_dir, f"saliency_batch{batch_idx:04d}.npy"),
            saliency.cpu().numpy(),
        )
        np.save(
            os.path.join(out_dir, f"spatio_spectral_scores_batch{batch_idx:04d}.npy"),
            ss_scores,
        )

        # resolve names
        sample_names = resolve_names(labels_dict, B, batch_idx, (B, C, H, W))

        for b in tqdm(range(B), desc=f"Samples (batch {batch_idx})", leave=True, dynamic_ncols=True):
            ss    = ss_scores[b]
            order = np.argsort(ss)[::-1]
            topk  = order[:SAVE_TOPK_CHANNELS]

            name = sample_names[b]

            # --- Resolve sample_path (for m/z axis loading) ---
            sp_raw = None
            if isinstance(labels_dict, dict) and "sample_path" in labels_dict:
                vals = labels_dict["sample_path"]
                sp_raw = vals[b] if isinstance(vals, (list, tuple)) else vals
                sp_raw = None if sp_raw is None else str(sp_raw)
            if sp_raw is None:
                sp_raw = name_to_sample_path(name)

            # Only now create per-sample directory & do all the work
            sample_dir = os.path.join(samples_root, name)
            os.makedirs(sample_dir, exist_ok=True)

            # --- Recover raw Condition label ---
            cond_label_str = None
            cond_binary    = None
            try:
                if "Condition" in label_maps and "Condition" in labels_dict:
                    cond_idx = labels_dict["Condition"][b]
                    if hasattr(cond_idx, "item"):
                        cond_idx = int(cond_idx.item())
                    cond_label_str = inv_label_maps["Condition"].get(cond_idx, None)
                    cond_binary    = map_condition_to_binary(cond_label_str)
            except Exception:
                cond_label_str = None
                cond_binary    = None

            # --- m/z axis loading ---
            mz_axis = None
            if sp_raw is not None:
                npz_path = resolve_npz_path(sp_raw)
                if npz_path:
                    mz_axis = load_mz_axis_only(npz_path)

            use_mz = mz_axis is not None and len(mz_axis) == ss.shape[0]

            # --- Save per-sample top-k table ---
            if use_mz:
                mz_top = mz_axis[topk]
                df_b = pd.DataFrame({
                    "rank":                   np.arange(1, len(topk) + 1),
                    "channel_index":          topk,
                    "mz":                     mz_top,
                    "spatio_spectral_score":  ss[topk],
                    "patch_embed_weight_norm":ch_weight_norm[topk],
                    "Condition_raw":          cond_label_str,
                    "Condition_binary":       cond_binary,
                    "sample_name":            name,
                })
            else:
                df_b = pd.DataFrame({
                    "rank":                   np.arange(1, len(topk) + 1),
                    "channel_index":          topk,
                    "spatio_spectral_score":  ss[topk],
                    "patch_embed_weight_norm":ch_weight_norm[topk],
                    "Condition_raw":          cond_label_str,
                    "Condition_binary":       cond_binary,
                    "sample_name":            name,
                })
            df_b.to_csv(os.path.join(sample_dir, "top_channels.csv"), index=False)

            # --- PCA RGB + saliency overview (for figures) ---
            img_chw = x[b].detach().cpu().numpy()
            save_pca_rgb(
                img_chw,
                os.path.join(sample_dir, "pca_rgb.png"),
                title=f"PCA-RGB | {name}",
            )
            sal_chw = saliency[b].detach().cpu().numpy()
            attn_hw = attn_maps[b, 0].detach().cpu().numpy()   # [H, W]

            save_saliency_heatmap(
                sal_chw,
                os.path.join(sample_dir, "saliency.png"),
                title=f"Saliency | {name}",
                attn_hw=attn_hw,     # <--- enables the joint map
                save_joint=True
            )

            # --- Top-k channel images ---
            if SAVE_TOPK_CHANNEL_IMS:
                for r, ch in enumerate(topk, start=1):
                    ch_img = img_chw[ch]
                    if use_mz:
                        mz_val = float(mz_axis[ch])
                        mz_str = _fmt_mz_for_filename(mz_val)
                        out_ch = os.path.join(sample_dir, f"top_{r:02d}_mz_{mz_str}.png")
                        title  = f"Top {r}: m/z {mz_val:.2f}"
                    else:
                        out_ch = os.path.join(sample_dir, f"top_{r:02d}_c{int(ch)}.png")
                        title  = f"Top {r}: channel {int(ch)}"
                    save_channel_viridis(ch_img, out_ch, title=title)

            # --- Master rows for global aggregation ---
            for rank, ch in enumerate(topk):
                row = {
                    "batch":                  batch_idx,
                    "sample_name":            name,
                    "sample_idx_in_batch":    b,
                    "rank":                   rank + 1,
                    "channel":                int(ch),
                    "spatio_spectral_score":  float(ss[ch]),
                    "patch_embed_weight_norm":float(ch_weight_norm[ch]),
                    "Condition_raw":          cond_label_str,
                    "Condition_binary":       cond_binary,
                }
                if use_mz:
                    row["mz"] = float(mz_axis[ch])
                all_rows.append(row)

        processed_batches += 1

        # ------------------ GLOBAL AGGREGATIONS ------------------
        if not all_rows:
            print("[WARN] No rows collected, nothing to aggregate.")
            return

        df_all = pd.DataFrame(all_rows)
        df_all.to_csv(
            os.path.join(out_dir, "spatio_spectral_topk_all_healthy_vs_diseased.csv"),
            index=False,
        )

        # ---- Per-channel mean (unchanged) ----
        agg_ch = (
            df_all.groupby("channel")["spatio_spectral_score"]
            .mean()
            .sort_values(ascending=False)
        )
        agg_ch_df = agg_ch.reset_index().rename(
            columns={"spatio_spectral_score": "mean_spatio_spectral_score"}
        )
        agg_ch_df.to_csv(
            os.path.join(out_dir, "spatio_spectral_channel_means.csv"),
            index=False,
        )

        # ------------------
        # m/z BINNING + AGGREGATION
        # ------------------
        MZ_BIN_WIDTH = 0.1  # Da; keep in sync with plotting script

        if "mz" in df_all.columns:
            # Safely convert to numeric (NaN if bad) and create a binned mz axis
            mz_vals = pd.to_numeric(df_all["mz"], errors="coerce")

            df_all["mz_bin"] = np.nan
            mask = mz_vals.notna() & np.isfinite(mz_vals.to_numpy())

            df_all.loc[mask, "mz_bin"] = (
                (mz_vals[mask] / MZ_BIN_WIDTH).round() * MZ_BIN_WIDTH
            ).round(6)

            # ---- overall m/z-bin stats ----
            df_mz = df_all.dropna(subset=["mz_bin"]).copy()

            if not df_mz.empty:
                agg_mz = (
                    df_mz.groupby("mz_bin")["spatio_spectral_score"]
                    .mean()
                    .sort_values(ascending=False)
                )
                agg_mz_df = agg_mz.reset_index().rename(
                    columns={
                        "mz_bin": "mz",
                        "spatio_spectral_score": "mean_spatio_spectral_score",
                    }
                )
                agg_mz_df.to_csv(
                    os.path.join(out_dir, "spatio_spectral_mz_means_overall.csv"),
                    index=False,
                )

                # ---- Healthy vs Diseased per m/z-bin ----
                if "Condition_binary" in df_mz.columns:
                    df_cb = df_mz.dropna(subset=["Condition_binary"]).copy()
                    if not df_cb.empty:
                        agg_mz_cond = (
                            df_cb.groupby(["Condition_binary", "mz_bin"])["spatio_spectral_score"]
                            .mean()
                            .reset_index()
                            .rename(
                                columns={
                                    "mz_bin": "mz",
                                    "spatio_spectral_score": "mean_spatio_spectral_score",
                                }
                            )
                        )
                        agg_mz_cond.to_csv(
                            os.path.join(
                                out_dir,
                                "spatio_spectral_mz_means_by_binary_condition.csv",
                            ),
                            index=False,
                        )

                        # wide-format pivot: rows = m/z-bin, columns = Healthy/Diseased
                        pivot = agg_mz_cond.pivot(
                            index="mz", columns="Condition_binary",
                            values="mean_spatio_spectral_score"
                        )
                        pivot = pivot.sort_index()

                        if "Healthy" in pivot.columns and "Diseased" in pivot.columns:
                            pivot["delta_Diseased_minus_Healthy"] = (
                                pivot["Diseased"] - pivot["Healthy"]
                            )

                        pivot.to_csv(
                            os.path.join(
                                out_dir,
                                "spatio_spectral_mz_means_binary_condition_pivot.csv",
                            )
                        )

        print(f"[DONE] Saved Healthy vs Diseased m/z attribution results (HVD-only) to: {out_dir}")

if __name__ == "__main__":
    main()

In [ ]:
import os
import shutil

# === USER CONFIG ===
OUT_DIR = r"attn_attrib_hvD/vit_base_patch14_dinov2__multitask_dinov2_final"  # <-- put your run dir here
SAMPLES_DIR = os.path.join(OUT_DIR, "samples")
PCA_RGB_DIR = os.path.join(OUT_DIR, "pca_rgb")  # sibling to 'samples'
os.makedirs(PCA_RGB_DIR, exist_ok=True)

for sample in os.listdir(SAMPLES_DIR):
    sample_dir = os.path.join(SAMPLES_DIR, sample)
    if not os.path.isdir(sample_dir):
        continue

    # look for pca_rgb.png inside each sample folder
    cand = os.path.join(sample_dir, "pca_rgb.png")
    if not os.path.isfile(cand):
        # optionally also allow variations like 'pca_rgb_*.png'
        pngs = [f for f in os.listdir(sample_dir)
                if f.lower().startswith("pca_rgb") and f.lower().endswith(".png")]
        if not pngs:
            print(f"[skip] no pca_rgb image in {sample}")
            continue
        cand = os.path.join(sample_dir, pngs[0])

    dst = os.path.join(PCA_RGB_DIR, f"{sample}_pca_rgb.png")
    shutil.copy2(cand, dst)
    print(f"[copied] {cand} -> {dst}")

print("\n✓ Collected PCA-RGBs in:", PCA_RGB_DIR)

### Plots

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# -------------------------
# Paths
# -------------------------
OUT_DIR  = r"attn_attrib_hvD/vit_base_patch14_dinov2__multitask_dinov2_final"
CSV_ALL  = os.path.join(OUT_DIR, "spatio_spectral_topk_all_healthy_vs_diseased.csv")

# -------------------------
# Load and clean
# -------------------------
df = pd.read_csv(CSV_ALL)
df = df.dropna(subset=["Condition_binary", "mz", "spatio_spectral_score"]).copy()
df["mz"] = pd.to_numeric(df["mz"], errors="coerce")
df = df.dropna(subset=["mz"])
df = df[df["Condition_binary"].isin(["Healthy", "Diseased"])]

# -------------------------
# Histogram binning
# -------------------------
BIN_WIDTH = 0.5
bins = np.arange(df["mz"].min(), df["mz"].max() + BIN_WIDTH, BIN_WIDTH)
bin_centers = (bins[:-1] + bins[1:]) / 2.0

H = df[df["Condition_binary"] == "Healthy"]
D = df[df["Condition_binary"] == "Diseased"]

hist_H, _ = np.histogram(H["mz"], bins=bins, weights=H["spatio_spectral_score"])
hist_D, _ = np.histogram(D["mz"], bins=bins, weights=D["spatio_spectral_score"])

# Normalize
hist_H = hist_H / hist_H.sum() if hist_H.sum() > 0 else hist_H
hist_D = hist_D / hist_D.sum() if hist_D.sum() > 0 else hist_D

# -------------------------
# Mask zero bins to prevent baseline lines
# -------------------------
mask_H = hist_H > 0
mask_D = hist_D > 0

# -------------------------
# Plot two bar histograms (overlaid bars)
# -------------------------
sns.set_theme(style="white", context="paper")
fig, ax = plt.subplots(figsize=(3.7, 3.0), dpi=400)

bar_width = BIN_WIDTH * 0.9

ax.bar(
    bin_centers[mask_H],
    hist_H[mask_H],
    width=5,
    alpha=0.55,
    color="#1f77b4",
    edgecolor="none",
    label="Healthy",
)

ax.bar(
    bin_centers[mask_D],
    hist_D[mask_D],
    width=5,
    alpha=0.55,
    color="#d95f02",
    edgecolor="none",
    label="Diseased",
)

# Labels
ax.set_xlabel("m/z", fontsize=9)
ax.set_ylabel("Normalized attribution density", fontsize=9)
ax.tick_params(axis="both", labelsize=8, direction="out")

sns.despine(ax=ax, top=True, right=True)
ax.legend(frameon=False, fontsize=7, loc="upper right")

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "mz_axis_two_hist_bars_pubready.png"),
            dpi=400, bbox_inches="tight", transparent=True)
plt.savefig(os.path.join(OUT_DIR, "mz_axis_two_hist_bars_pubready.svg"),
            bbox_inches="tight", transparent=True)
plt.show()